[![Run in Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/atsyplenkov/ai_geoimage_coreg/blob/main/examples/default_coreg_colab.ipynb)

# AI GeoImage Coreg - Default Colab Run

**Recommended runtime:** Google Colab with **GPU (T4)**.

It is assumed that inputs and outputs in this workflow are stored on **Google Drive** (mounted to `/content/drive`).

## 1) Verify GPU Runtime

If this cell fails, switch runtime to **Runtime -> Change runtime type -> T4 GPU** and rerun.

In [ ]:
import torch

print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    raise RuntimeError('GPU is not enabled. In Colab: Runtime -> Change runtime type -> T4 GPU.')

## 2) Mount Google Drive

When the Drive authorization prompt pops up, approve it to connect Google Colab to your Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Required Input Files

Upload your two input GeoTIFF files to `DRIVE_ROOT` (or update the paths below):

- Historical image: `historical.tif`
- Georeferenced modern reference image: `modern.tif`

Both files are expected to be on **Google Drive**.

## 3) Set Input and Output Paths on Drive

Define where your historical image, modern reference image, and outputs live on Drive.

In [ ]:
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/ai_geoimage_coreg')
HISTORICAL_IMAGE = DRIVE_ROOT / 'historical.tif'
REFERENCE_IMAGE = DRIVE_ROOT / 'modern.tif'
OUTPUT_PREFIX = DRIVE_ROOT / 'georef'

DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print(f'DRIVE_ROOT: {DRIVE_ROOT}')
print('If your files are elsewhere in Drive, edit DRIVE_ROOT/HISTORICAL_IMAGE/REFERENCE_IMAGE above.')

## 4) Install Dependencies in Colab

Install GDAL system packages and the latest repository code from GitHub.

In [ ]:
!apt-get update
!apt-get install -y gdal-bin libgdal-dev
!pip install --upgrade pip
!pip install git+https://github.com/atsyplenkov/ai_geoimage_coreg.git

## 5) Validate Imports

Check that GDAL, Rasterio, and `run_pipeline` import correctly before processing.

In [ ]:
from osgeo import gdal
import rasterio
from ai_geoimage_coreg.core import run_pipeline

print('GDAL version:', gdal.VersionInfo())
print('Rasterio version:', rasterio.__version__)
print('Imports OK. If these imports fail after install, restart runtime and rerun cells.')

## 6) Confirm Input Files Exist

Check that both input TIFF files are present before running the pipeline.

In [ ]:
missing = [str(p) for p in [HISTORICAL_IMAGE, REFERENCE_IMAGE] if not p.exists()]
if missing:
    raise FileNotFoundError(
        'Missing input files:\n'
        + '\n'.join(missing)
        + '\n\nUpload both TIFFs to DRIVE_ROOT or edit the path variables above.'
    )

print('Input files found.')

## 7) Run Default Coregistration

Run the default pipeline and write outputs to Google Drive.

In [ ]:
run_pipeline(
    path_hex=str(HISTORICAL_IMAGE),
    path_ref=str(REFERENCE_IMAGE),
    output_prefix=str(OUTPUT_PREFIX),
)

## 8) Check Output Files

Verify that expected result files are present in your Drive output folder.

In [ ]:
expected_outputs = [
    DRIVE_ROOT / 'georef_raw.csv',
    DRIVE_ROOT / 'georef_clean.csv',
    DRIVE_ROOT / 'georef_poly.tif',
    DRIVE_ROOT / 'georef_tps.tif',
]

for out in expected_outputs:
    print(f'{out.name}: {"OK" if out.exists() else "MISSING"}')